# MT-ViT on Kaggle: prepare the data, fine-tune the model, measure before and after
**Before you press Run All:** Settings → Accelerator **GPU T4 x2** (or P100) · Internet **On** · inputs A, B and C added (see README).
Takes about 1 to 1.5 hours. When it finishes, download **model.zip** and **results.zip** from the Output panel.

In [ ]:
# 1. Get our code and the one library Kaggle is missing
!rm -rf /kaggle/working/code && git clone -q https://github.com/vidush-islockedin/DeepFake-detection /kaggle/working/code
!pip install -q facenet-pytorch==2.6.0 --no-deps
%cd /kaggle/working/code
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings")

In [ ]:
# 2. Find the three datasets wherever Kaggle mounted them (fast: walks folders only, never the image folders)
import os
from pathlib import Path
SKIP = {"real", "fake", "train_sample_videos", "test_videos", "faces"}   # folders full of images/videos
def find(pattern, required=True, root="/kaggle/input", max_depth=6):
    level = [Path(root)]
    for _ in range(max_depth + 1):
        for d in level:                                  # shallowest match wins
            if (d / pattern).exists():
                return str(d / pattern)
        nxt = []
        for d in level:
            try:
                nxt += [Path(e.path) for e in os.scandir(d) if e.is_dir() and e.name.lower() not in SKIP]
            except OSError:
                pass
        level = sorted(nxt)
    assert not required, f"Not found: {pattern}. Add that dataset with '+ Add Input'. Mounted: {os.listdir(root)}"
    return None
A = Path(find("Dataset/Train")).parent              # manjilkarki/deepfake-and-real-images
B = Path(find("real-vs-fake/train")).parent         # xhlulu/140k-real-and-fake-faces
C = find("train_sample_videos/metadata.json", required=False)   # DFDC (competition deepfake-detection-challenge)
print("A:", A, "\nB:", B, "\nC:", C or "not added - v2 needs it, add the competition with '+ Add Input'")

In [ ]:
# 3. MTCNN face crops -> /kaggle/temp/faces/<train|val|test>/<A|B|C>/<real|fake>/
FACES = "/kaggle/temp/faces"
!python prepare.py --src "{A}" --name A --out {FACES} --per-class 5000
!python prepare.py --src "{B}" --name B --out {FACES} --per-class 5000
if C:
    !python prepare.py --dfdc "{C}" --name C --out {FACES} --dfdc-frames 4   # 60% of DFDC identities go to training
!for d in {FACES}/*/*/*; do echo "$d $(ls $d | wc -l)"; done

In [ ]:
# 4. BEFORE: the downloaded pre-trained model, untouched
!python evaluate.py --model dima806/deepfake_vs_real_image_detection --data {FACES}/test --out /kaggle/working/results/before

In [ ]:
# 5. Fine-tune on A + B + the training part of C (augmented, balanced, cut-off tuned on validation)
!python train.py --data {FACES} --out /kaggle/working/model --epochs 3 --bs 32

In [ ]:
# 6. AFTER: our fine-tuned model on the same test sets
!python evaluate.py --model /kaggle/working/model --data {FACES}/test --out /kaggle/working/results/after
import json
for name in ("before", "after"):
    r = json.load(open(f"/kaggle/working/results/{name}/results.json"))
    print(name.upper(), "cut-off", r["threshold"], {k: (round(v["accuracy"], 4), "FP", v["real_called_fake"], "FN", v["fake_called_real"]) for k, v in r.items() if isinstance(v, dict)})

In [ ]:
# 7. Pack the two files to download
%cd /kaggle/working
!cp model/history.csv model/threshold.json results/ && zip -qr model.zip model && zip -qr results.zip results && ls -lh *.zip